# Psychometric and chronometric curves per trial mode (supplementary)

For each of the 17 trial modes (fig_engagement_syllables D–F): the probability of a rightward choice and the median reaction time against signed contrast, split by block. Each mode keeps its **number and colour** from fig_engagement_syllables (numbered 1..17 in colour-wheel order of the UMAP map, `umap_pastel` palette): the swatch above each pair of panels.

* **Trials:** the trial-mode fit `3_trial_modes/trial_modes_8_k_10_bin_syllables_02-10-2026`, joined on (session, trial_id) to the trial-meta file for contrast side, block, choice and reaction time (every joined trial's feedback / contrast / block / choice agree with the mode table's own `trial_type`).
* **Left and right are oriented on the stimulus and the block, not on the `choice` strings**, which are mirrored in every trial table of this repo (a 'right' label is a report of a LEFT stimulus). Signed contrast = right − left contrast (in the p(left) = 0.2 block, 80 % of stimuli are on the positive side); a rightward choice = the `'left'` label.
* **Statistic:** per mouse, the proportion of rightward choices (or median RT, trials with RT in `RT_WINDOW`) in each mode × block × contrast cell with ≥ `MIN_TRIALS` trials; then mean ± SEM across mice, drawn where ≥ `MIN_MICE` mice contribute. The share of all trials each mode accounts for (in the whole mode fit, as fig_engagement_syllables E) is printed in its swatch. Contrast on a linear axis; RT on a log axis, so fast and slow modes share one scale.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from datetime import date

FIGDIR = pathlib.Path.cwd().resolve()
while not (FIGDIR / 'fig_layout.py').exists() and FIGDIR != FIGDIR.parent:
    FIGDIR = FIGDIR / 'figure_generation' if (FIGDIR / 'figure_generation').exists() else FIGDIR.parent
PI = FIGDIR.parent
REPO = PI.parent
for p in [str(REPO), str(PI), str(FIGDIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)
from Functions import trial_modes as tmodes
import paper_style as ps
import fig_layout as fl
ps.use('paper')
FIG = 'fig_trial_mode_curves'

MODES_FILE = PI / '3_trial_modes' / 'trial_modes_8_k_10_bin_syllables_02-10-2026' / 'trials.pqt'
TRIAL_META = next(f for f in [PI / 'data' / 'session_trial_meta_19-08-2026', PI / 'data' / 'session_trial_meta_10-07-2026']
                  if f.exists())
MIN_TRIALS = 3          # trials a mouse needs in a mode x block x contrast cell
MIN_MICE = 3            # mice a point needs to be drawn
RT_WINDOW = (0, 2)      # s, as fig_engagement
BLOCKS = [0.2, 0.5, 0.8]
N_COLS = 6

## Trials, modes and their colours

In [ ]:
trials = pd.read_parquet(MODES_FILE)
ORDER = [int(k) for k in tmodes.umap_order(trials)]              # cluster ids in colour-wheel order
MODE_NAME = {k: i + 1 for i, k in enumerate(ORDER)}              # cluster id -> displayed number (as fig_engagement_syllables)
MODE_COLORS = tmodes.umap_colors(trials, lightness=76, chroma=40)    # the 'umap_pastel' palette
ink_on = lambda c: 'white' if ps.hex_to_oklch(c)[0] < 0.62 else 'k'

meta = pd.read_parquet(TRIAL_META, columns=['session', 'trial_id', 'choice', 'feedback', 'block', 'contrast',
                                            'signed_contrast', 'reaction'])
T = (trials[['session', 'trial_id', 'mouse_name', 'trial_type', 'trial_cluster']].dropna(subset=['trial_cluster'])
     .merge(meta.assign(trial_id=meta['trial_id'].astype(float)), on=['session', 'trial_id'], how='inner'))
T['mode'] = T['trial_cluster'].astype(int)
parts = T['trial_type'].str.split(expand=True)
assert ((parts[0] == T['feedback']) & (parts[1].astype(float) == T['contrast'])
        & (parts[2].astype(float) == T['block']) & (parts[3] == T['choice'])).all(), 'mode table and trial meta disagree'
T = T[T['choice'].isin(['left', 'right'])].copy()
T['right'] = (T['choice'] == 'left').astype(float)                 # the labels are mirrored: 'left' = rightward report
T['cx'] = (T['signed_contrast'] * 100).round(2)                    # % ; > 0 = stimulus on the right
T['rt'] = T['reaction'].where(T['reaction'].between(*RT_WINDOW))
# share of ALL trials in the mode fit (every proficient session), as fig_engagement_syllables E
SHARE = trials['trial_cluster'].dropna().astype(int).value_counts(normalize=True)
print('displayed number <- cluster id:', {v: k for k, v in MODE_NAME.items()})
# orientation check: correct trials with a stimulus on the right are rightward reports
_c = T[(T['feedback'] == 'correct') & (T['cx'] != 0)]
print(f"{len(T)} trials, {T['session'].nunique()} sessions, {T['mouse_name'].nunique()} mice "
      f"(meta: {TRIAL_META.name}); check -- correct & stimulus right -> rightward: "
      f"{(_c['right'] == (_c['cx'] > 0)).mean():.3f}")

## Per mouse, then across mice

In [ ]:
def curves(value):
    d = T.dropna(subset=[value])
    cell = (d.groupby(['mode', 'block', 'cx', 'mouse_name'])[value]
             .agg(['mean' if value == 'right' else 'median', 'size']).reset_index())
    cell.columns = ['mode', 'block', 'cx', 'mouse_name', 'v', 'n']
    cell = cell[cell['n'] >= MIN_TRIALS]
    out = cell.groupby(['mode', 'block', 'cx'])['v'].agg(['mean', 'sem', 'size']).reset_index()
    return out[out['size'] >= MIN_MICE]

PSY, CHRONO = curves('right'), curves('rt')
print(PSY.groupby('mode')['size'].median().rename(index=MODE_NAME).sort_index().rename('median mice per point').to_string())

## The figure

In [ ]:
XT = [-100, -50, 0, 50, 100]

def cx_axis(ax, labels=True):
    # linear contrast axis (the low contrasts, 6.25 and 12.5 %, sit close to 0)
    ax.set_xticks(XT); ax.set_xticklabels(['-100', '', '0', '', '100'] if labels else [])
    ax.set_xlim(-110, 110)
    ax.axvline(0, color='0.85', lw=0.5, zorder=0)

def mode_cell(fig, spec, k, first_col, last_row):
    g = spec.subgridspec(3, 1, height_ratios=[0.22, 1, 1], hspace=0.12)
    sw = fig.add_subplot(g[0]); sw.set_axis_off()
    c = MODE_COLORS[float(k)]
    sw.add_patch(Rectangle((0, 0), 1, 1, transform=sw.transAxes, color=c, lw=0))
    sw.text(0.5, 0.5, f"mode {MODE_NAME[k]}   {100 * SHARE.get(k, 0):.1f}%", transform=sw.transAxes, ha='center',
            va='center', color=ink_on(c), fontsize=plt.rcParams['font.size'] * 0.85, fontweight='bold')
    ap, ac = fig.add_subplot(g[1]), fig.add_subplot(g[2])
    for b in BLOCKS:
        col = ps.BLOCK[b]
        p = PSY[(PSY['mode'] == k) & (PSY['block'] == b)].sort_values('cx')
        ap.errorbar(p['cx'], p['mean'], yerr=p['sem'], color=col, lw=0.8, ms=1.8, marker='o', capsize=0, elinewidth=0.6)
        r = CHRONO[(CHRONO['mode'] == k) & (CHRONO['block'] == b)].sort_values('cx')
        ac.errorbar(r['cx'], r['mean'], yerr=r['sem'], color=col, lw=0.8, ms=1.8, marker='o', capsize=0, elinewidth=0.6)
    cx_axis(ap, labels=False); cx_axis(ac, labels=last_row)
    ap.set_ylim(-0.03, 1.03); ap.set_yticks([0, 0.5, 1]); ap.axhline(0.5, color='0.85', lw=0.5, zorder=0)
    ac.set_yscale('log'); ac.set_ylim(0.08, 1.5); ac.set_yticks([0.1, 0.3, 1]); ac.set_yticklabels(['0.1', '0.3', '1'])
    ac.yaxis.set_minor_locator(plt.NullLocator())
    if first_col:
        ap.set_ylabel('P(right)'); ac.set_ylabel('RT (s)')
    else:
        ap.tick_params(labelleft=False); ac.tick_params(labelleft=False)
    if last_row:
        ac.set_xlabel('Contrast (%)', labelpad=1)
    return [sw, ap, ac]

n_rows = int(np.ceil((len(ORDER) + 1) / N_COLS))
fig = plt.figure(figsize=(fl.FIG_WIDTH, 1.95 * n_rows))
G = fig.add_gridspec(n_rows, N_COLS, wspace=0.18, hspace=0.3, left=0.07, right=0.99, top=0.97, bottom=0.07)
for i, k in enumerate(ORDER):
    r_, c_ = divmod(i, N_COLS)
    last = (i + N_COLS >= len(ORDER))                              # nothing below it
    mode_cell(fig, G[r_, c_], k, first_col=(c_ == 0), last_row=last)
# the free slot: the block legend
ax = fig.add_subplot(G[divmod(len(ORDER), N_COLS)]); ax.set_axis_off()
ax.legend(handles=[Line2D([], [], color=ps.BLOCK[b], marker='o', ms=2.5, lw=0.8, label=ps.BLOCK_LABEL[b]) for b in BLOCKS],
          title='Block', frameon=False, loc='center')
plt.show()
OUT = FIGDIR / 'figures'
for path in [OUT / f"{FIG}_{date.today().strftime('%d-%m-%Y')}.svg", OUT / f'{FIG}.pdf', OUT / f'{FIG}.png']:
    fig.savefig(path, dpi=300)
    print('wrote', path.relative_to(PI))